# Backtesting Strategies - Advanced Guide

This notebook demonstrates how to backtest different scoring strategies using historical data.

In [ ]:
import pandas as pd
import sys
from pathlib import Path
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from finviz_weekly.backtest import Backtester, BacktestConfig, compare_strategies

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Check Available History

First, verify you have sufficient historical data for backtesting.

In [ ]:
history_path = Path("../data/history/finviz_scored_history.parquet")

if history_path.exists():
    history = pd.read_parquet(history_path)
    history["as_of_date"] = pd.to_datetime(history["as_of_date"])
    
    print(f"Total rows: {len(history)}")
    print(f"Date range: {history['as_of_date'].min()} to {history['as_of_date'].max()}")
    print(f"Unique dates: {history['as_of_date'].nunique()}")
    print(f"Unique tickers: {history['ticker'].nunique()}")
    
    print("\nAvailable score columns:")
    score_cols = [col for col in history.columns if col.startswith("score_")]
    for col in sorted(score_cols):
        print(f"  - {col}")
else:
    print("⚠️ History file not found. Run weekly scraping to build history.")

## 2. Single Strategy Backtest

Test a single scoring strategy.

In [ ]:
if history_path.exists():
    # Create backtester
    backtester = Backtester(history_path)
    
    # Configure strategy
    config = BacktestConfig(
        start_date="2024-01-01",
        end_date="2025-01-01",
        score_column="score_quality_value",  # Change this to test different strategies
        top_n=20,
        rebalance_days=7,
    )
    
    # Run backtest
    print("Running backtest...")
    results = backtester.run(config)
    
    # Display results
    print("\n" + "="*60)
    print("BACKTEST RESULTS")
    print("="*60)
    print(f"Strategy: {config.score_column}")
    print(f"Period: {config.start_date} to {config.end_date}")
    print(f"Portfolio: Top {config.top_n} stocks, rebalance every {config.rebalance_days} days")
    print("")
    print(f"Total Return:    {results.total_return:>8.1%}")
    print(f"Annual Return:   {results.annual_return:>8.1%}")
    print(f"Sharpe Ratio:    {results.sharpe_ratio:>8.2f}")
    print(f"Max Drawdown:    {results.max_drawdown:>8.1%}")
    print(f"Trades:          {len(results.trades):>8}")
    print("="*60)
else:
    print("Cannot run backtest without history data")

### Visualize Portfolio Value Over Time

In [ ]:
if history_path.exists() and 'results' in locals():
    plt.figure(figsize=(12, 6))
    plt.plot(results.portfolio_values.index, results.portfolio_values.values, linewidth=2)
    plt.title(f"Portfolio Value Over Time - {config.score_column}", fontsize=14)
    plt.xlabel("Date")
    plt.ylabel("Portfolio Value ($)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Calculate and plot drawdown
    cummax = results.portfolio_values.cummax()
    drawdown = (results.portfolio_values - cummax) / cummax
    
    plt.figure(figsize=(12, 4))
    plt.fill_between(drawdown.index, drawdown.values * 100, 0, alpha=0.3, color='red')
    plt.plot(drawdown.index, drawdown.values * 100, color='red', linewidth=1)
    plt.title("Drawdown Over Time (%)", fontsize=14)
    plt.xlabel("Date")
    plt.ylabel("Drawdown (%)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 3. Compare Multiple Strategies

Test multiple strategies side-by-side.

In [ ]:
if history_path.exists():
    # Define strategies to compare
    strategies = [
        ("Master", "score_master"),
        ("Quality Value", "score_quality_value"),
        ("Compounders", "score_compounders"),
        ("GARP", "score_garp"),
    ]
    
    # Add enhanced strategies if available
    if "score_insider_momentum" in history.columns:
        strategies.append(("Insider Momentum", "score_insider_momentum"))
    if "score_earnings_surprise" in history.columns:
        strategies.append(("Earnings Surprise", "score_earnings_surprise"))
    if "score_enhanced_master" in history.columns:
        strategies.append(("Enhanced Master", "score_enhanced_master"))
    
    print(f"Comparing {len(strategies)} strategies...\n")
    
    # Run comparison
    comparison_df = compare_strategies(
        history_path,
        start_date="2024-01-01",
        end_date="2025-01-01",
        strategies=strategies,
        top_n=20,
        rebalance_days=7
    )
    
    # Format and display
    comparison_df["total_return"] = (comparison_df["total_return"] * 100).round(1)
    comparison_df["annual_return"] = (comparison_df["annual_return"] * 100).round(1)
    comparison_df["sharpe_ratio"] = comparison_df["sharpe_ratio"].round(2)
    comparison_df["max_drawdown"] = (comparison_df["max_drawdown"] * 100).round(1)
    
    # Sort by Sharpe ratio
    comparison_df = comparison_df.sort_values("sharpe_ratio", ascending=False)
    
    print("\nStrategy Comparison (sorted by Sharpe Ratio):")
    display(comparison_df)
else:
    print("Cannot compare strategies without history data")

### Visualize Strategy Comparison

In [ ]:
if history_path.exists() and 'comparison_df' in locals():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Total Return
    axes[0, 0].barh(comparison_df["strategy"], comparison_df["total_return"])
    axes[0, 0].set_xlabel("Total Return (%)")
    axes[0, 0].set_title("Total Return Comparison")
    axes[0, 0].grid(True, alpha=0.3, axis='x')
    
    # Sharpe Ratio
    axes[0, 1].barh(comparison_df["strategy"], comparison_df["sharpe_ratio"], color='green')
    axes[0, 1].set_xlabel("Sharpe Ratio")
    axes[0, 1].set_title("Sharpe Ratio Comparison")
    axes[0, 1].grid(True, alpha=0.3, axis='x')
    
    # Max Drawdown
    axes[1, 0].barh(comparison_df["strategy"], comparison_df["max_drawdown"], color='red')
    axes[1, 0].set_xlabel("Max Drawdown (%)")
    axes[1, 0].set_title("Maximum Drawdown")
    axes[1, 0].grid(True, alpha=0.3, axis='x')
    
    # Number of Trades
    axes[1, 1].barh(comparison_df["strategy"], comparison_df["num_trades"], color='orange')
    axes[1, 1].set_xlabel("Number of Trades")
    axes[1, 1].set_title("Trading Activity")
    axes[1, 1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

## 4. Parameter Optimization

Test different portfolio sizes and rebalancing frequencies.

In [ ]:
if history_path.exists():
    # Test different portfolio sizes
    print("Testing different portfolio sizes...")
    
    results_by_size = []
    for top_n in [10, 15, 20, 30, 50]:
        config = BacktestConfig(
            start_date="2024-01-01",
            end_date="2025-01-01",
            score_column="score_master",
            top_n=top_n,
            rebalance_days=7,
        )
        result = backtester.run(config)
        results_by_size.append({
            "top_n": top_n,
            "total_return": result.total_return * 100,
            "sharpe_ratio": result.sharpe_ratio,
            "max_drawdown": result.max_drawdown * 100,
        })
    
    size_df = pd.DataFrame(results_by_size)
    print("\nPortfolio Size Optimization:")
    display(size_df)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].plot(size_df["top_n"], size_df["total_return"], marker='o')
    axes[0].set_xlabel("Portfolio Size (# of stocks)")
    axes[0].set_ylabel("Total Return (%)")
    axes[0].set_title("Return vs Portfolio Size")
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(size_df["top_n"], size_df["sharpe_ratio"], marker='o', color='green')
    axes[1].set_xlabel("Portfolio Size (# of stocks)")
    axes[1].set_ylabel("Sharpe Ratio")
    axes[1].set_title("Sharpe Ratio vs Portfolio Size")
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Cannot optimize parameters without history data")

In [ ]:
if history_path.exists():
    # Test different rebalancing frequencies
    print("Testing different rebalancing frequencies...")
    
    results_by_rebal = []
    for rebal_days in [1, 3, 7, 14, 30]:
        config = BacktestConfig(
            start_date="2024-01-01",
            end_date="2025-01-01",
            score_column="score_master",
            top_n=20,
            rebalance_days=rebal_days,
        )
        result = backtester.run(config)
        results_by_rebal.append({
            "rebalance_days": rebal_days,
            "total_return": result.total_return * 100,
            "sharpe_ratio": result.sharpe_ratio,
            "num_trades": len(result.trades),
        })
    
    rebal_df = pd.DataFrame(results_by_rebal)
    print("\nRebalancing Frequency Optimization:")
    display(rebal_df)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].plot(rebal_df["rebalance_days"], rebal_df["sharpe_ratio"], marker='o', color='green')
    axes[0].set_xlabel("Rebalance Frequency (days)")
    axes[0].set_ylabel("Sharpe Ratio")
    axes[0].set_title("Sharpe Ratio vs Rebalance Frequency")
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(rebal_df["rebalance_days"], rebal_df["num_trades"], marker='o', color='orange')
    axes[1].set_xlabel("Rebalance Frequency (days)")
    axes[1].set_ylabel("Number of Trades")
    axes[1].set_title("Trading Activity vs Rebalance Frequency")
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Cannot optimize rebalancing without history data")

## 5. Trade Analysis

In [ ]:
if history_path.exists() and 'results' in locals() and len(results.trades) > 0:
    trades_df = results.trades
    
    print(f"Total trades: {len(trades_df)}")
    print(f"Buy trades: {(trades_df['action'] == 'BUY').sum()}")
    print(f"Sell trades: {(trades_df['action'] == 'SELL').sum()}")
    
    print("\nTop 10 Largest Trades (by shares):")
    display(trades_df.nlargest(10, "shares"))
    
    print("\nMost Traded Tickers:")
    most_traded = trades_df["ticker"].value_counts().head(10)
    display(pd.DataFrame(most_traded))
else:
    print("No trade data available")

## Key Takeaways

1. **Build History First**: Collect 3-6 months of enhanced data for meaningful backtests
2. **Compare Strategies**: Test multiple approaches to find what works best
3. **Optimize Parameters**: Portfolio size and rebalancing frequency significantly impact returns
4. **Consider Transaction Costs**: More frequent rebalancing = higher costs
5. **Validate on Different Periods**: Test across bull, bear, and sideways markets

## Next Steps

- Collect more historical data for robust backtests
- Test enhanced strategies once sufficient data is available
- Optimize factor weights based on backtest results
- Consider transaction costs in strategy selection